# 01 — Compression ratio linearly predicts benchmark score (on a small open-model sample)

**Programme 01 — Compression-as-Intelligence.** [Programme file.](../programmes/01-compression-as-intelligence.md)

By the end of this notebook you will have computed bits-per-byte on a fixed text sample for 4 small open language models, paired each model with a fixed-task benchmark score, fit a linear regression, and seen — on this tiny sample — the qualitative shape that Huang et al. (2024, [arXiv:2404.09937](https://arxiv.org/abs/2404.09937)) measured across 31 models. The claim being demonstrated is the linear coupling between average-corpus compression performance and downstream capability.

What this notebook is *not*: a serious test of Huang et al. We use 4 models, ~1KB of text, and a single benchmark. A 4-point regression is statistical theater for pedagogy. The qualitative shape is the takeaway, not the $R^2$.

In [ ]:
# Install pinned versions (Colab cell; skip locally if your env already has them).
# %pip install -q torch==2.4.1 transformers==4.45.2 matplotlib==3.9.2 scikit-learn==1.5.2 numpy==1.26.4 datasets==3.0.1

In [ ]:
import math, os, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
os.makedirs('figures/01', exist_ok=True)

## A fixed text sample

We use a short English-text sample. In real evaluations (Huang et al., Delétang et al.) the sample is much larger; here we use a tiny one for budget. The choice of corpus matters: a code corpus would re-order models very differently.

In [ ]:
SAMPLE_TEXT = (
    'The Roman Empire was the post-Republican period of ancient Rome. As a polity, it included large territorial holdings around the Mediterranean Sea in Europe, North Africa, and West Asia, and was ruled by emperors. From the accession of Caesar Augustus as the first Roman emperor to the military anarchy of the 3rd century, it was a Principate with Italia as the metropole of its provinces and the city of Rome as its sole capital. The Empire was later ruled by multiple emperors who shared control over the Western Roman Empire and the Eastern Roman Empire. The city of Rome was the largest city in the world c. 100 BC and c. AD 400, with Constantinople (New Rome) becoming the largest around 500 AD, and the Empire\'s populace grew to an estimated 50 to 90 million inhabitants (roughly 20% of the world\'s population at the time). The 500-year-old republic which preceded it was severely destabilized in a series of civil wars and political conflicts. In the mid-1st century BC, Julius Caesar was appointed as perpetual dictator and then assassinated in 44 BC. Civil wars and proscriptions continued, eventually resulting in the victory of Octavian, Caesar\'s adopted son, over Mark Antony and Cleopatra at the Battle of Actium in 31 BC, and the subsequent annexation of the Ptolemaic Kingdom in Egypt. In 27 BC, the Roman Senate granted Octavian overarching military power and the new title of Augustus, marking his accession as the first Roman emperor of a now unified Roman Empire.'
)
BYTES = SAMPLE_TEXT.encode('utf-8')
print(f'corpus length: {len(BYTES)} bytes')

## Bits-per-byte via cross-entropy

For each model we compute the model's average negative log-likelihood per token in nats, convert to bits, and divide by *bytes* (not tokens), so the metric is tokenizer-invariant. This is the standard bpb definition used in Delétang et al. (2023) and elsewhere.

**Prediction (before running):** bpb decreases from `gpt2` to `gpt2-medium` to `pythia-410m` to `pythia-1.4b`. Benchmark scores increase in the same order.

In [ ]:
MODELS = [
    'gpt2',
    'gpt2-medium',
    'EleutherAI/pythia-410m',
    'EleutherAI/pythia-1.4b',
]

@torch.no_grad()
def bits_per_byte(model_name: str, text: str) -> float:
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32).to(DEVICE).eval()
    ids = tok(text, return_tensors='pt').input_ids.to(DEVICE)
    out = model(ids, labels=ids)
    nll_per_token_nats = out.loss.item()              # mean across tokens
    n_tokens = ids.shape[1]
    n_bytes = len(text.encode('utf-8'))
    total_bits = nll_per_token_nats * n_tokens / math.log(2)
    bpb = total_bits / n_bytes
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return bpb

bpb_values = []
for m in MODELS:
    v = bits_per_byte(m, SAMPLE_TEXT)
    bpb_values.append(v)
    print(f'{m:>32s}  bpb = {v:.3f}')

## Benchmark scores

We hard-code four benchmark scores from public reports (HellaSwag accuracy, normalized). In a serious evaluation you would pull these from a held-out evaluation you ran yourself, or from an authoritative leaderboard snapshot. These numbers are rounded approximations of values reported around 2023; treat them as illustrative.

In [ ]:
HELLASWAG_APPROX = {
    'gpt2':                    0.289,
    'gpt2-medium':             0.333,
    'EleutherAI/pythia-410m':  0.397,
    'EleutherAI/pythia-1.4b':  0.518,
}
bench = np.array([HELLASWAG_APPROX[m] for m in MODELS])
bpb = np.array(bpb_values)
for m, b, h in zip(MODELS, bpb, bench):
    print(f'{m:>32s}  bpb={b:.3f}  hellaswag~{h:.3f}')

## Linear fit and plot

**Prediction:** $R^2$ should be substantially above zero, and the slope negative (lower bpb → higher score). With $n=4$ the value is suggestive, not confirmatory.

In [ ]:
X = bpb.reshape(-1, 1)
y = bench
reg = LinearRegression().fit(X, y)
r2 = reg.score(X, y)
print(f'slope    = {reg.coef_[0]:+.4f}')
print(f'intercept = {reg.intercept_:+.4f}')
print(f'R^2      = {r2:.3f}')

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.scatter(bpb, bench, s=64)
xs = np.linspace(bpb.min() * 0.95, bpb.max() * 1.05, 50)
ax.plot(xs, reg.predict(xs.reshape(-1, 1)), linestyle='--')
for x_, y_, m in zip(bpb, bench, MODELS):
    ax.annotate(m.split('/')[-1], (x_, y_), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('bits-per-byte (lower = better compression)')
ax.set_ylabel('approx. HellaSwag accuracy')
ax.set_title(f'compression vs benchmark on 4 small open LMs (R^2={r2:.2f})')
fig.tight_layout()
fig.savefig('figures/01/bpb_vs_hellaswag.png', dpi=120)
plt.show()

## What this shows. What this does not show. What would refute it.

**What this shows.** On a 4-model sample, bits-per-byte on a Wikipedia-style English snippet correlates with HellaSwag accuracy in the expected direction. The slope is negative and the magnitudes are roughly what you would extrapolate from Huang et al.'s 31-model fit.

**What this does not show.** With $n=4$, the regression has zero statistical power. We hard-coded benchmark numbers from public reports and used a tiny corpus; either choice could be swapped to make the fit better or worse. We have not separated within-family from across-family variance. We have not tested reasoning-finetuned models, which are the obvious failure case for the linearity claim.

**What would refute the claim at this scale.** A reasoning-finetuned model whose bpb is *similar* to its base model but whose HellaSwag score is *substantially* higher (or vice versa) would visually break the line and would be a fair small-scale challenge to the linear coupling.

**Where to go next.** Read Huang et al. (2024) and Delétang et al. (2023). The serious evidence is at 31-model scale and on multiple benchmarks; this notebook is the lowest-budget way to internalize the variables.